This notebook performs advanced analysis on input text, including:
- Sentiment scoring
- Emotion detection
- Cognitive distortion detection
- Linguistic feature extraction (pronouns, intensity, absolutist language)
- Readability score
- Psychological markers (e.g., helplessness, agency reduction)
- Report generation and simple tests

In [1]:
import re
import math
from collections import Counter

Lexicons

In [2]:
Positive = ["good", "great", "happy", "love", "calm", "capable", "hopeful"]
Negative = ["bad", "sad", "angry", "hate", "terrible", "worthless", "broken", "failure"]


Emotions = {
"sadness": ["sad", "unhappy", "down", "depressed", "lonely"],
"anger": ["angry", "furious", "mad", "irritated"],
"fear": ["afraid", "scared", "anxious", "worried"],
"joy": ["happy", "joyful", "excited", "glad"],
"shame": ["ashamed", "embarrassed", "guilty"]
}


Distortions = {
"all_or_nothing": r"\b(always|never|everytime|nothing|completely)\b", #a regex method " r"\b\b" to extract words from text"
"catastrophizing": r"\b(disaster|ruined|terrible|awful|collapse)\b",
"mind_reading": r"\b(they think|everyone thinks|people believe)\b",
"overgeneralization": r"\b(everyone|nobody|everything)\b",
"negative_self_talk": r"\b(i am (stupid|worthless|a failure|broken))\b",
"emotional_reasoning": r"\b(i feel (bad|terrible|awful),? so it must be true)\b"
}

Readibility level calculator

In [3]:
def read_score(text):
    words = text.split()
    if not words:
        return 0.0
    sentences = max(1, text.count('.') + text.count('!') + text.count('?'))
    #vowels per word
    vow_count = sum(sum(1 for ch in word if ch in 'aeiou') for word in words)
    # avoid division by zero
    words_per_sentence = len(words) / sentences
    vow_per_word = vow_count / len(words) if len(words) else 0
    score = 206.835 - 1.015 * words_per_sentence - 84.6 * vow_per_word
    return round(score, 2)
    

Psychological markers

In [4]:
def psych_markers(words):
    #Counting linguistic markers often used in psychological text analysis
    markers = {}
    first_person = [w for w in words if w in ["i", "me", "my", "mine"]]
    absolutist = [w for w in words if w in ["always", "never", "nothing", "completely"]]
    helplessness = [w for w in words if w in ["can't", "cannot", "unable", "cant"]]
    
    markers["self_focus"] = len(first_person)
    markers["absolutist_terms"] = len(absolutist)
    markers["helplessness_terms"] = len(helplessness)
    # pronoun ratio counter
    markers["total_words"] = len(words)
    markers["first_person_ratio"] = round(len(first_person) / (len(words) or 1), 3)
    return markers

Analysis function

In [5]:
def analyze_text(text):
    text = (text or "").strip()
    text_lower = text.lower()
    # finding the same word occurace in text
    words = re.findall(r"\b[\w']+\b", text_lower)
    # Sentiment
    pos = sum(1 for w in words if w in Positive)
    neg = sum(1 for w in words if w in Negative)
    if pos > neg:
        sentiment = "positive"
    elif neg > pos:
        sentiment = "negative"
    else:
        sentiment = "neutral"
    #Emotions
    feelings = {emo: sum(1 for w in words if w in lex) for emo, lex in Emotions.items()}
    max_val = max(feelings.values()) if feelings else 0
    dominant = [emo for emo, v in feelings.items() if v == max_val and v > 0]
    if not dominant:
        dominant = ["neutral"]
    #distortions
    dist = [name for name, pattern in Distortions.items() if re.search(pattern, text_lower)]
    # Add psych markers
    markers = psych_markers(words)
    # Readability
    reading_ease = read_score(text)
    return {
        "sentiment": sentiment,
        "positive_count": pos,
        "negative_count": neg,
        "emotion_count": feelings,
        "dominant_emotion": dominant,
        "cognitive_distortions": dist,
        "linguistic_markers": markers,
        "readability_score": reading_ease,
    
            }

Report generation

In [8]:
def rep_gen(analysis):
    report = []
    report.append("Psychological Analysis Report")
    report.append("Emotion Counts:")
    for emo, count in analysis.get('emotion_count', {}).items():
        report.append(f" - {emo}: {count}")
    report.append(f"Dominant Emotion: {', '.join(analysis.get('dominant_emotion', ['neutral']))}")
    report.append("")
    report.append("Cognitive Distortions Detected:")
    if analysis.get('cognitive_distortions'):
        for d in analysis.get('cognitive_distortions'):
            report.append(f" - {d}")
    else:
        report.append(" - None detected")
    report.append("")
    report.append("Linguistic Markers:")
    for m, val in analysis.get('linguistic_markers', {}).items():
        report.append(f" - {m}: {val}")
    report.append("")
    report.append(f"Readability Score: {analysis.get('readability_score')}")
    return "\n".join(report)

User Test

In [16]:
if __name__ == "__main__":
# interactive example
    try:
        text = input("Enter text: ")
    except Exception:
        text = "I will and can do everything to get out of this mess."
    result = analyze_text(text)
    print(rep_gen(result))
    #additional text tests
    tests = [
        ("I am so excited for tommorrow!", "hope"),
        ("I always mess things up.", "sadness"),
        ("I'm anxious about getting job.", "fear"),
            ]

    print("\n--- Running quick test cases ---")
    for idx, (ttext, expected_emotion) in enumerate(tests, 1):
        out = analyze_text(ttext)
        dom = out['dominant_emotion'][0]
        status = "PASS" if expected_emotion == dom or expected_emotion in dom else "WARN"
        print(f"{idx}. expected: {expected_emotion:7} | detected: {dom:7} | {status}")

    print("\nNotebook analysis complete.") 

Enter text:  I will and can do everything to get out of this mess.


Psychological Analysis Report
Emotion Counts:
 - sadness: 0
 - anger: 0
 - fear: 0
 - joy: 0
 - shame: 0
Dominant Emotion: neutral

Cognitive Distortions Detected:
 - overgeneralization

Linguistic Markers:
 - self_focus: 1
 - absolutist_terms: 0
 - helplessness_terms: 0
 - total_words: 12
 - first_person_ratio: 0.083

Readability Score: 95.95

--- Running quick test cases ---
1. expected: hope    | detected: joy     | WARN
2. expected: sadness | detected: neutral | WARN
3. expected: fear    | detected: fear    | PASS

Notebook analysis complete.
